In [ ]:
# 1. Download data
!gdown --id '1e4CaQ5VUF3F04XRDGXrnRQGogo89TiF8' --output real_or_drawing.zip
!unzip real_or_drawing.zip

In [ ]:
# 2. Visualize the data
import matplotlib.pyplot as plt


def no_axis_show(img, title="", cmap=None):
    # imshow, and set the interpolation mode to be "nearest"。
    fig = plt.imshow(img, interpolation="nearest", cmap=cmap)
    # do not show the axes in the images.
    fig.axes.get_xaxis().set_visible(False)
    fig.axes.get_yaxis().set_visible(False)
    plt.title(title)


titles = [
    "horse",
    "bed",
    "clock",
    "apple",
    "cat",
    "plane",
    "television",
    "dog",
    "dolphin",
    "spider",
]
plt.figure(figsize=(18, 18))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    fig = no_axis_show(
        plt.imread(f"real_or_drawing/train_data/{i}/{500 * i}.bmp"), title=titles[i]
    )

In [ ]:
plt.figure(figsize=(18, 18))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    fig = no_axis_show(
        plt.imread(f"real_or_drawing/test_data/0/" + str(i).rjust(5, "0") + ".bmp")
    )

因為大家塗鴉的時候通常只會畫輪廓，我們可以根據這點將source data做點邊緣偵測處理，讓source data更像target data一點。

算法這邊不贅述，只教大家怎麼用。若有興趣歡迎參考wiki或這裡。

cv2.Canny使用非常方便，只需要兩個參數: low_threshold, high_threshold。

cv2.Canny(image, low_threshold, high_threshold)

簡單來說就是當邊緣值超過high_threshold，我們就確定它是edge。如果只有超過low_threshold，那就先判斷一下再決定是不是edge。

以下我們直接拿source data做做看。

In [ ]:
# 3. Special Domain Knowledge
import cv2
import matplotlib.pyplot as plt

titles = [
    "horse",
    "bed",
    "clock",
    "apple",
    "cat",
    "plane",
    "television",
    "dog",
    "dolphin",
    "spider",
]
original_img = plt.imread("real_or_drawing/train_data/0/0.bmp")
gray_img = cv2.cvtColor(original_img, cv2.COLOR_RGB2GRAY)

plt.figure(figsize=(18, 18))

plt.subplot(1, 5, 1)
no_axis_show(original_img, title="original")

plt.subplot(1, 5, 2)
no_axis_show(gray_img, title="gray scale", cmap="gray")

CANNY_LOW, CANNY_HIGH = 170, 300
canny_img = cv2.Canny(gray_img, CANNY_LOW, CANNY_HIGH)
plt.subplot(1, 5, 3)
no_axis_show(canny_img, title=f"Canny({CANNY_LOW}, {CANNY_HIGH})", cmap="gray")

for idx, (low, high) in enumerate([(50, 100), (150, 200), (250, 300)], start=3):
    canny = cv2.Canny(gray_img, low, high)
    plt.subplot(1, 5, idx + 1)
    no_axis_show(canny, title=f"Canny({low}, {high})", cmap="gray")
plt.show()

在這裡我故意將data用成可以使用torchvision.ImageFolder的形式，所以只要使用該函式便可以做出一個datasets。

transform的部分請參考以下註解。

In [ ]:
# 4. Data Process
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function

import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

source_transform = transforms.Compose(
    [
        transforms.Grayscale(),
        transforms.Lambda(lambda x: cv2.Canny(np.array(x), 170, 300)),
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15, fill=(0,)),
        transforms.ToTensor(),
    ]
)
target_transform = transforms.Compose(
    [
        transforms.Grayscale(),
        transforms.Resize((32, 32)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15, fill=(0,)),
        transforms.ToTensor(),
    ]
)

source_dataset = ImageFolder("real_or_drawing/train_data", transform=source_transform)
target_dataset = ImageFolder("real_or_drawing/test_data", transform=target_transform)

source_dataloader = DataLoader(source_dataset, batch_size=32, shuffle=True)
target_dataloader = DataLoader(target_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(target_dataset, batch_size=128, shuffle=False)

Feature Extractor: 典型的VGG-like疊法。

Label Predictor / Domain Classifier: MLP到尾。

相信作業寫到這邊大家對以下的Layer都很熟悉，因此不再贅述。

In [ ]:
# 5. Model
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, 1, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        x = self.conv(x).squeeze(-1).squeeze(-1)  # 或是 x = torch.flatten(x, start_dim=1)
        return x


class LabelPredictor(nn.Module):
    def __init__(self):
        super(LabelPredictor, self).__init__()

        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, h):
        c = self.layer(h)
        return c


class DomainClassifier(nn.Module):
    def __init__(self):
        super(DomainClassifier, self).__init__()

        self.layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, h):
        y = self.layer(h)
        return y

In [ ]:
# 6. Pre-processing
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

LR = 1e-4
EPOCHS = 200
LAMBDA = 0.1
BATCH_SIZE = 32
CHECKPOINT_DIR = "checkpoints"

feature_extractor = FeatureExtractor().to(device)
label_predictor = LabelPredictor().to(device)
domain_classifier = DomainClassifier().to(device)

class_criterion = nn.CrossEntropyLoss()
domain_criterion = nn.BCEWithLogitsLoss()

optimizer_F = optim.Adam(feature_extractor.parameters(), lr=LR)
optimizer_C = optim.Adam(label_predictor.parameters(), lr=LR)
optimizer_D = optim.Adam(domain_classifier.parameters(), lr=LR)

# 新增 Scheduler
scheduler_F = optim.lr_scheduler.StepLR(optimizer_F, step_size=50, gamma=0.5)
scheduler_C = optim.lr_scheduler.StepLR(optimizer_C, step_size=50, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(optimizer_D, step_size=50, gamma=0.5)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

理論上，在原始paper中是加上Gradient Reversal Layer，並將Feature Extractor / Label Predictor / Domain Classifier 一起train，但其實我們也可以交換的train Domain Classfier & Feature Extractor(就像在train GAN的Generator & Discriminator一樣)，這也是可行的。

在code實現中，我們採取後者的方式。

小提醒
原文中的lambda(控制Domain Adversarial Loss的係數)是有Adaptive的版本，如果有興趣可以參考原文。
因為我們完全沒有target的label，所以結果如何，只好丟kaggle看看囉:)?

In [ ]:
# 7. Start Training
def train_epoch(source_dataloader, target_dataloader, lamb):
    """
    Args:
      source_dataloader: source data的dataloader
      target_dataloader: target data的dataloader
      lamb: control the balance of domain adaptatoin and classification.
    """

    # D loss: Domain Classifier的loss
    # F loss: Feature Extrator & Label Predictor的loss
    running_D_loss = running_F_loss = total_hit = total_num = 0.0

    for i, ((source_data, source_label), (target_data, _)) in enumerate(
        tqdm(
            zip(source_dataloader, target_dataloader),
            total=min(len(source_dataloader), len(target_dataloader)),
        )
    ):
        source_data, source_label = source_data.to(device), source_label.to(device)
        target_data = target_data.to(device)

        mixed_data = torch.cat([source_data, target_data], dim=0)
        domain_label = torch.zeros(mixed_data.size(0), 1).to(device)
        domain_label[: source_data.size(0)] = 1

        # Step 1 : train domain classifier
        optimizer_D.zero_grad()
        feature = feature_extractor(mixed_data).detach()
        # We don't need to train feature extractor in step 1.
        # Thus we detach the feature neuron to avoid backpropgation.
        domain_logits = domain_classifier(feature)
        loss_D = domain_criterion(domain_logits, domain_label)
        running_D_loss += loss_D.item()
        loss_D.backward()
        optimizer_D.step()

        # Step 2 : train feature extractor and label classifier
        optimizer_F.zero_grad()
        optimizer_C.zero_grad()
        feature = feature_extractor(mixed_data)
        class_logits = label_predictor(feature[: source_data.shape[0]])
        domain_logits = domain_classifier(feature)
        # loss = cross entropy of classification - lamb * domain binary cross entropy.
        # The reason why using subtraction is similar to generator loss in disciminator of GAN
        loss_F = class_criterion(class_logits, source_label) - lamb * domain_criterion(
            domain_logits, domain_label
        )
        loss_F.backward()
        optimizer_F.step()
        optimizer_C.step()
        running_F_loss += loss_F.item()

        total_hit += (torch.argmax(class_logits, dim=1) == source_label).sum().item()
        total_num += source_data.size(0)

    return running_D_loss / (i + 1), running_F_loss / (i + 1), total_hit / total_num

In [ ]:
# 8. Validation
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

val_ratio = 0.1
source_indices = list(range(len(source_dataset)))
train_idx, val_idx = train_test_split(
    source_indices,
    test_size=val_ratio,
    random_state=42,
    stratify=source_dataset.targets,
)

source_train_dataset = Subset(source_dataset, train_idx)
source_val_dataset = Subset(source_dataset, val_idx)

source_train_loader = DataLoader(
    source_train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False)


def validate(feature_extractor, label_predictor, val_loader, device):
    feature_extractor.eval()
    label_predictor.eval()
    total_hit, total_num = 0, 0

    with torch.no_grad():
        for data, label in val_loader:
            data, label = data.to(device), label.to(device)
            features = feature_extractor(data)
            logits = label_predictor(features)
            total_hit += (torch.argmax(logits, dim=1) == label).sum().item()
            total_num += label.size(0)

    return total_hit / total_num


# 驗證後恢復訓練模式的輔助函數（可選）
def set_train_mode(models):
    for model in models:
        model.train()

In [ ]:
# 9. 主訓練迴圈
best_train_acc = 0.0
best_val_acc = 0.0
patience = 20
no_improve_epochs = 0
early_stop = False

for epoch in range(EPOCHS):
    if early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

    # 訓練階段
    feature_extractor.train()
    label_predictor.train()
    domain_classifier.train()

    train_D_loss, train_F_loss, train_acc = train_epoch(
        source_train_loader, target_dataloader, LAMBDA
    )

    # 驗證階段
    feature_extractor.eval()
    label_predictor.eval()
    domain_classifier.eval()

    val_acc = validate(feature_extractor, label_predictor, source_val_loader, device)

    scheduler_F.step()
    scheduler_C.step()
    scheduler_D.step()
    
    if train_acc > best_train_acc:
        best_train_acc = train_acc
        torch.save(
            {
                "epoch": epoch,
                "feature_extractor": feature_extractor.state_dict(),
                "label_predictor": label_predictor.state_dict(),
                "optimizer_F": optimizer_F.state_dict(),
                "optimizer_C": optimizer_C.state_dict(),
                "best_train_acc": train_acc,
            },
            f"{CHECKPOINT_DIR}/best_train_acc.pt"
        )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve_epochs = 0
        torch.save(
            {
                "epoch": epoch,
                "feature_extractor": feature_extractor.state_dict(),
                "label_predictor": label_predictor.state_dict(),
                "optimizer_F": optimizer_F.state_dict(),
                "optimizer_C": optimizer_C.state_dict(),
                "best_val_acc": best_val_acc,
            },
            f"{CHECKPOINT_DIR}/best_val_acc.pt",
        )
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            early_stop = True

    if (epoch + 1) % 10 == 0:
        torch.save(
            {
                "epoch": epoch,
                "feature_extractor": feature_extractor.state_dict(),
                "label_predictor": label_predictor.state_dict(),
                "optimizer_F": optimizer_F.state_dict(),
                "optimizer_C": optimizer_C.state_dict(),
            },
            f"{CHECKPOINT_DIR}/latest.pt",
        )

    print(
        f"epoch {epoch:3d}: D_loss={train_D_loss:.4f}, F_loss={train_F_loss:.4f}, "
        f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}"
    )

In [ ]:
# 10. 
import numpy as np
import torch
from tqdm import tqdm


def calibrate_by_distribution(logits, eps=1e-8, target_dist=None):
    """
    利用測試集類別平衡做後處理校正
    logits: shape (N, 10) 的模型輸出 logits
    target_dist: 目標分佈，預設為均勻 [0.1, ..., 0.1]
    """
    if target_dist is None:
        target_dist = np.ones(10) / 10.0

    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    pred_dist = probs.mean(axis=0)

    adjustment = np.log(pred_dist + eps)
    calibrated_logits = logits - adjustment

    calibrated_preds = np.argmax(calibrated_logits, axis=1)
    return calibrated_preds, pred_dist


# 若想更穩定，可加入以下版本，把上面 calibrate_by_distribution 換成 calibrate_iterative(all_logits) 即可。
def calibrate_iterative(logits, max_iter=5, eps=1e-8):
    """迭代式校正，直到分佈接近均勻"""
    current_logits = logits.copy().astype(np.float64)
    for _ in range(max_iter):
        probs = torch.softmax(torch.from_numpy(current_logits), dim=1).numpy()
        pred_dist = probs.mean(axis=0)
        adjustment = np.log(pred_dist + eps)
        current_logits = current_logits - adjustment
    return np.argmax(current_logits, axis=1)

In [ ]:
# 11. Inference + Strong Baseline Calibration
label_predictor.eval()
feature_extractor.eval()

all_logits = []
all_probs = []

with torch.no_grad():
    for test_data, _ in tqdm(test_dataloader, desc="Inference"):
        test_data = test_data.to(device)
        features = feature_extractor(test_data)
        class_logits = label_predictor(features)

        all_logits.append(class_logits.cpu().numpy())

all_logits = np.concatenate(all_logits, axis=0)

final_preds, pred_dist = calibrate_by_distribution(all_logits)

print("預測類別分佈（校正前平均機率）:", np.round(pred_dist, 4))
print("校正後類別計數:", np.bincount(final_preds, minlength=10))

import pandas as pd

df = pd.DataFrame({"id": np.arange(len(final_preds)), "label": final_preds})
df.to_csv("DaNN_submission_strong_baseline.csv", index=False)
print("已輸出 DaNN_submission_strong_baseline.csv")

8. Training Statistics
Number of parameters:<br>

Feature Extractor: 2, 142, 336<br>
Label Predictor: 530, 442<br>
Domain Classifier: 1, 055, 233